In [1]:
import xmlrpc.client as xml
import os
import csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import traceback
import math
import sys
import time

proxy = xml.ServerProxy("http://localhost:1080/RPC2")
plecs = proxy.plecs
directory = r"C:\Users\Goat_\Documents\Research\Class E\\"
model = "derated compression"
plecs.load(directory + model + ".plecs")

datadir = directory + model + "/PWM_freq_data/"
os.makedirs(datadir, exist_ok=True)

In [2]:
def get_data(file):
    data = pd.read_csv(directory + file)
    return data

In [3]:
def round_sig(x, sig=4):
    if x == 0:
        return 0.0
    return round(x, sig - int(math.floor(math.log10(abs(x)))) - 1)

In [15]:
freq = np.linspace(800e3, 1200e3, num=21, endpoint=True)
print("Freq:", freq)

pwm = np.linspace(0.1, 0.9, num=17, endpoint=True)
print("PWM:", pwm)

loads = np.array([0, 5000, 10000, 15000, 20000, 25000, 30000, 50000, 70000, 10e6])
print("Loads:", loads)

Freq: [ 800000.  820000.  840000.  860000.  880000.  900000.  920000.  940000.
  960000.  980000. 1000000. 1020000. 1040000. 1060000. 1080000. 1100000.
 1120000. 1140000. 1160000. 1180000. 1200000.]
PWM: [0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65 0.7  0.75
 0.8  0.85 0.9 ]
Loads: [0.0e+00 5.0e+03 1.0e+04 1.5e+04 2.0e+04 2.5e+04 3.0e+04 5.0e+04 7.0e+04
 1.0e+07]


In [16]:
filename = datadir + "3VIN_4TURN_data.csv"
if not os.path.exists(filename):
    with open(filename, "w") as f:
        writer = csv.writer(f)
        write_to_csv = ["kOhm", "kHz", "PWM", "IIN", "PIN", "IRMS", "VRMS", "POUT", "EFF", "ZVS"]
        writer.writerow(write_to_csv)

In [18]:
curr_time = time.time()

for l in loads:
    for f in freq:
        for p in pwm:
            if p < 0.5:
                opts = {'ModelVars' : { 
                    'rload' : float(l),
                    'freq' : float(f),
                    'phasem' : -float(p) * 2,
                    'phasep' : 1.0
                } }
            if p >= 0.5:
                opts = {'ModelVars' : { 
                    'rtest' : float(l),
                    'freq' : float(f),
                    'phasem' : -1.0,
                    'phasep' : float(p) * 2
                } }
            
            try:
                results = plecs.simulate(model, opts)
    
                IIN = round_sig(results['Values'][2][-1], 4)
                PIN = round_sig(results['Values'][3][-1], 4)
                IRMS = round_sig(results['Values'][4][-1], 4)
                VRMS = round_sig(results['Values'][5][-1], 4)
                POUT = round_sig(results['Values'][6][-1], 4)
                EFF = round_sig(results['Values'][7][-1], 4)
                ZVS = round_sig(min(results['Values'][8][-200:]), 4)
            except Exception as e:
                IIN = 1e-9
                PIN = 1e-9
                IRMS = 1e-9
                VRMS = 1e-9
                POUT = 1e-9
                EFF = 1e-9
                ZVS = 1e-9

            elapsed = int(time.time() - curr_time)

            print(
                f"\rElapsed: {elapsed}  PWM: {p:.3f} F: {f/1e3:.1f} kHz, R: {l/1e3:.1f} kOhm  IIN: {IIN} PIN: {PIN} IRMS: {IRMS} VRMS: {VRMS} POUT: {POUT} EFF: {EFF} ZVS: {ZVS}",
                end="",
                flush=True
            )
            
            with open(filename, "a", newline="") as csvfile:
                writer = csv.writer(csvfile)
                write_to_csv = [l, f, p, IIN, PIN, IRMS, VRMS, POUT, EFF, ZVS]
                writer.writerow(write_to_csv)

Elapsed: 319  PWM: 0.150 F: 860.0 kHz, R: 0.0 kOhm  IIN: 5.66 PIN: 17.05 IRMS: 0.001251 VRMS: 4.38 POUT: 0.00548 EFF: 0.0003215 ZVS: -8.729.931


KeyboardInterrupt



# TESTING


In [ ]:
        opts = {'ModelVars' : { 
            'rload' : float(700e9),
            'freq' : float(1.085e6),
        } }

In [ ]:
results = plecs.simulate(model, opts)

In [ ]:
ZVS = min(results['Values'][8][:-10])
print(ZVS)

In [ ]:
print(min(results['Values'][8][-200:]))